# Seismic Refraction on Your Own Field Data

**Near-Surface Geophysics — Missouri S&T**

You have shot records from a refraction line: hammer shots into one spread of
geophones, fired from several positions along it — off the near end, at the
first geophone, part-way along, at the far geophone, off the far end. This
notebook turns those files into a layered velocity model, by the method you
would use on paper: pick the first arrival on every trace, plot travel time
against offset, find the slope and intercept of each straight branch, and
convert those into velocities and depths.

1. **Read** the records. Geometry, units and recording delay come out of the
   file headers.
2. **Pick** first breaks automatically, then correct them by hand. This is
   the slow part, and it is the part that decides the answer.
3. **Plot** every shot's picks together on one travel-time graph.
4. **Fit** two or three layers by the intercept-time method.
5. **Compare** model against data — and go back to step 2 when the residuals
   tell you to. That loop is the point of the notebook.

Units are **metres and seconds** throughout; plots are in milliseconds
because that is the scale a first break is judged on. A Geode exporting in
feet is converted on read, and the conversion is recorded in the survey's
provenance.

> **The one thing to take away.** The automatic picker writes a first draft.
> Every number printed below sits downstream of picks that you are
> responsible for having looked at. Section 4 exists so that you can look at
> them, and sections 7 and 8 exist so that you find out when you should have.

## 0. Setup

Run this cell first. It installs `shallowgeo` and the widget library, and
imports what the rest of the notebook uses.

In [ ]:
# The package is under active development, so this checks for the *refraction*
# code rather than just for `import shallowgeo`. A runtime that installed an
# older copy imports fine and then has no shallowgeo.refraction -- and because
# the version string does not change between builds, a plain `pip install`
# sees "already satisfied" and does nothing. Hence the forced second attempt.
import importlib
import sys

PACKAGE = ("shallow-geophysics[seismic] @ "
           "git+https://github.com/Maurer-GEMLab/shallow-geophysics.git")

def shallowgeo_ready():
    """True if shallowgeo.refraction can be imported, after dropping any
    stale copy the kernel already loaded."""
    for name in [n for n in sys.modules
                 if n == "shallowgeo" or n.startswith("shallowgeo.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    try:
        from shallowgeo.refraction import PickingSession  # noqa: F401
        return True
    except ImportError:
        return False

if not shallowgeo_ready():
    %pip install -q --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    %pip install -q --force-reinstall --no-deps --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    raise ImportError(
        "shallowgeo.refraction is still missing after installing. In Colab: "
        "Runtime > Restart session, then run this cell again."
    )

%pip install -q ipywidgets

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.axisbelow": True})

import shallowgeo as sg
from shallowgeo.refraction import (
    PickingSession,
    crossover_distances,
    fit_layers,
    load_shots,
    plot_traveltimes,
)
print(f"ready - shallow-geophysics {sg.__version__} from {Path(sg.__file__).parent}")

## 1. Get your shot records in

**Run the one option that applies to you**, so that `DATA_DIR` points at the
folder holding your shot files. SEG-2 (`.dat`) and SEG-Y (`.sgy`, `.segy`)
are both read; a Geode writes one or the other depending on how the
SeisModule Controller was set up.

In [ ]:
# --- Option A: upload a .zip of your shot records (Colab) ----------------
DATA_DIR = None
try:
    from google.colab import files
    import io, tempfile, zipfile
    print("Choose your .zip of shot records...")
    uploaded = files.upload()
    name = next(iter(uploaded))
    DATA_DIR = Path(tempfile.mkdtemp()) / "refraction"
    with zipfile.ZipFile(io.BytesIO(uploaded[name])) as z:
        z.extractall(DATA_DIR)
    print(f"extracted to {DATA_DIR}")
except ImportError:
    print("Not running in Colab - use Option B, C or D below.")

In [ ]:
# --- Option B: Google Drive (Colab) --------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/refraction-line')

# --- Option C: a local folder --------------------------------------------
# DATA_DIR = Path('~/data/my-refraction-line').expanduser()

# --- Option D: the worked example that ships with the repository ---------
# Twelve usable shots into a 28 m spread, plus one deliberately bad record.
# Running in Colab? Clone the repository first:
#     !git clone https://github.com/Maurer-GEMLab/shallow-geophysics.git
#     DATA_DIR = Path('shallow-geophysics/examples/data/2026-09-18-refraction-line/raw')
# DATA_DIR = Path('../examples/data/2026-09-18-refraction-line/raw')

In [ ]:
files_found = sorted(p for p in Path(DATA_DIR).rglob("*")
                     if p.suffix.lower() in {".dat", ".sgy", ".segy", ".sg2"}
                     and not p.name.startswith("._"))
assert files_found, f"no shot records found under {DATA_DIR}"
print(f"{len(files_found)} files")
for p in files_found:
    print("  ", p.name)

## 2. What is actually in the files

Read them and look at the geometry **before** picking anything. Three things
go wrong often enough to check every time:

- **Units.** A Geode set to feet writes feet into the headers. `shallowgeo`
  converts to metres on read; the table below shows which files needed it.
  A survey silently interpreted in the wrong unit gives velocities out by a
  factor of 3.28 and nothing else looks wrong.
- **Recording delay.** A non-zero delay means the first sample is not the
  trigger. It is carried on the time axis, so picks are already corrected —
  but if the delay is wrong in the header, every pick is wrong by the same
  amount, and that shows up later as a direct-wave branch that misses the
  origin.
- **More than one shot per file.** A Geode file can hold several field
  records. `load_shots` splits them into one gather per source and labels
  each with its field-record number, so a file holding two shots becomes two
  entries below. If you see one entry where you expect two, the shot count
  is wrong and every offset in the second record is wrong with it.

In [ ]:
# Drop a file here once the plots below show it is unusable, e.g. ["4011"].
EXCLUDE = []

shots = load_shots(files_found, exclude=EXCLUDE)
print(f"{len(shots)} shot gathers from {len(files_found)} files")

rows = []
for label, survey in shots.items():
    prov = survey.provenance[0].parameters
    rows.append({
        "shot": label,
        "source_x_m": float(survey.geometry.table.set_index(["role", "id"]).loc[
            ("source", survey.trace_map["source_id"].iloc[0]), "x"]),
        "traces": survey.n_traces,
        "dt_ms": 1e3 * survey.sample_interval,
        "record_ms": 1e3 * survey.duration,
        "delay_ms": 1e3 * survey.delay,
        "header_units": prov.get("header_units", "?"),
        "file": Path(str(survey.metadata.get("source_file", ""))).name,
    })
inventory = pd.DataFrame(rows).sort_values("source_x_m").reset_index(drop=True)
display(inventory.round(3))

In [ ]:
# The spread and the shot positions, drawn. Anything surprising here is a
# geometry problem, and no amount of careful picking will fix one.
first = next(iter(shots.values()))
rec_x = first.geometry.receivers["x"].to_numpy(float)
fig, ax = plt.subplots(figsize=(11, 2.6))
ax.plot(rec_x, np.zeros_like(rec_x), "v", ms=9, color="#256abf", label="geophones")
for label, survey in shots.items():
    sx = float(survey.geometry.table.set_index(["role", "id"]).loc[
        ("source", survey.trace_map["source_id"].iloc[0]), "x"])
    ax.plot(sx, 0.35, "*", ms=13, color="#eb6834")
    ax.annotate(label, (sx, 0.35), textcoords="offset points", xytext=(0, 9),
                ha="center", fontsize=7, rotation=45)
ax.plot([], [], "*", ms=13, color="#eb6834", label="shots")
ax.set_ylim(-0.4, 1.2); ax.set_yticks([])
ax.set_xlabel("position along the line (m)")
ax.set_title(f"{len(rec_x)} geophones at {np.diff(np.sort(rec_x)).min():.2f} m "
             f"spacing, {len(shots)} shots")
ax.legend(loc="upper right", fontsize=8); ax.grid(axis="x", alpha=0.3)
plt.show()

spread = rec_x.max() - rec_x.min()
print(f"spread length {spread:.1f} m. The intercept-time method resolves an "
      f"interface reliably to roughly a quarter to a third of that,\nso expect "
      f"useful depths down to about {spread / 4:.0f}-{spread / 3:.0f} m and "
      f"treat anything deeper as an extrapolation.")

## 3. First arrivals, automatically

The **first break** is the moment the earliest energy reaches a geophone —
the first departure from the pre-trigger noise, not the first large swing. On
a hammer record the first break at long offset is often small, while the
largest amplitude on the trace is a surface wave arriving much later. Pickers
that chase amplitude land on the surface wave; the default here, `"aic"`,
asks instead where the statistics of the trace change, which is the right
question to ask.

Two bounds keep it honest. **`min_time`** skips the trigger spike at time
zero. **`max_time`** is the one that decides the answer, and it is worth
getting right, so it is done in two passes.

**Pass 1** leaves `max_time` unset. The AIC picker then bounds each trace by
its own largest amplitude, which is a sensible default — the first break is
never later than the biggest arrival — but it is a per-trace bound with no
knowledge of the line as a whole, so a trace whose noise happens to peak late
can still be picked far too late.

In [ ]:
session = PickingSession(
    shots,
    method="aic",        # "aic", "mer" or "sta_lta"
    min_time=0.001,      # skip the trigger spike
    min_quality=10.0,    # picks below this energy ratio start out excluded
)
print(session)

first_pass = session.table()
fig, ax = plt.subplots(figsize=(9.5, 5))
plot_traveltimes(first_pass, ax=ax, legend=False)
ax.set_title("Pass 1: picks with no line-wide time bound")
plt.show()

**Pass 2** uses the line itself. The first arrival at a given offset can never
be later than the **direct wave**, $x / v_0$ — every refracted path is a
short cut, which is the whole reason head waves overtake. So once you have a
rough $v_0$, you have a hard ceiling on every pick on the line:

$$t_{\max} \;=\; \text{safety} \times \frac{x_{\max}}{v_0}.$$

The cell below reads $v_0$ off pass 1 — the median of offset over time for
the nearest third of the picks, which is close to the direct-wave slope
however scattered the far picks are — and re-picks with that ceiling. The
safety factor covers the error in that estimate.

This matters more than it sounds. On the example dataset, two mispicks near
100 ms on one record survive pass 1 and pull the refractor velocity from
3400 m/s to 18700 m/s, with the RMS misfit going from 1.6 ms to 6.8 ms. Two
picks out of 265.

In [ ]:
SAFETY = 1.5   # how much slack to leave on the direct-wave ceiling

near = first_pass.nsmallest(max(5, len(first_pass) // 3), "offset")
v0_estimate = float((near["offset"] / near["time"]).median())
max_offset = float(first_pass["offset"].max())
MAX_TIME = SAFETY * max_offset / v0_estimate

print(f"direct wave from the near picks: about {v0_estimate:.0f} m/s")
print(f"longest offset {max_offset:.1f} m -> direct arrival at "
      f"{1e3 * max_offset / v0_estimate:.0f} ms")
print(f"ceiling for every pick on the line: {1e3 * MAX_TIME:.0f} ms")

session.auto_pick(max_time=MAX_TIME)
print("\n", session)

In [ ]:
# Every record with its picks on it. Look at all of them. A pick line that
# steps smoothly from trace to trace is believable; one that jumps by several
# milliseconds between neighbouring geophones is not, because the ground
# cannot change that fast over one geophone spacing.
labels = list(session.labels)
ncols = 3
nrows = -(-len(labels) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.2 * nrows),
                         squeeze=False)
for ax, label in zip(axes.ravel(), labels):
    session.plot_shot(label, ax=ax, tmax=min(1.3 * MAX_TIME, first.duration))
    ax.legend().remove()
for ax in axes.ravel()[len(labels):]:
    ax.axis("off")
fig.tight_layout()
plt.show()

## 4. Correct the picks by hand

This is the work. The panel below shows one record at a time with its picks,
and the selected trace enlarged beside it.

- **Choose a shot**, then step through the channels with the slider or the
  `<` `>` buttons.
- **Move a pick** with the *pick (ms)* slider, watching the enlarged trace.
- **`no pick here`** for a trace you cannot pick — a dead geophone, or the
  one at the source point that the hammer blow saturated. Dropping a trace is
  a legitimate answer and is better than guessing.
- **`keep this pick`** includes or excludes a pick without moving it. Use it
  to re-admit a low-quality pick that is actually correct.
- **`drop whole shot`** rejects an entire bad record.
- Under **automatic picker** you can change the method or the time bounds and
  **re-pick**. Your hand corrections survive that — they are only lost if you
  ask for it.

**What you are looking for.** The first break is where the trace first leaves
the flat pre-arrival noise, at the *start* of the first swing, not at its
peak and not at the first zero crossing after it. Pick the same feature on
every trace: a systematic bias of a millisecond matters far less than a
scatter of a millisecond, because a bias mostly shifts the intercept while
scatter corrupts the slope.

**When the picks look wrong on a whole shot**, suspect the record rather than
the picker, and check it against another shot fired from the same position if
you have one.

In [ ]:
# OPTIONAL: click-to-pick instead of sliders. Needs the ipympl backend.
# If this errors, or the plots go blank, skip it and use the sliders --
# they do the same job. To undo: run  %matplotlib inline  and re-run section 4.
#
# %pip install -q ipympl
# try:
#     from google.colab import output
#     output.enable_custom_widget_manager()
# except ImportError:
#     pass
# %matplotlib widget

In [ ]:
session.widget(tmax=min(1.3 * MAX_TIME, first.duration))

### Where the picks stand

One row per shot. The columns to read across:

- **`picks_used`** well below `traces` means the quality filter threw a lot
  away. Go back and look at why.
- **`max_time_ms`** far out of line with the other shots is the signature of
  a record whose picks have run away onto a later arrival — especially if
  another shot was fired from the same `source_x` and disagrees.
- **`median_quality`** near the threshold across a whole shot means a weak
  record, not a bad picker.

In [ ]:
display(session.summary().round(2))

In [ ]:
# Save the picks. This is the file to keep -- re-running the notebook
# re-picks from scratch, and loading this back restores your corrections.
PICKS_CSV = Path("first_break_picks.csv")
session.save(PICKS_CSV)
print(f"wrote {len(session.table(used_only=False))} picks to {PICKS_CSV.resolve()}")

# To restore them in a later session, after building `session` as above:
#   session.load(PICKS_CSV)

## 5. Every shot on one travel-time graph

Now the picks come together. Travel time against **offset** — the absolute
source-to-geophone distance — is the domain the interpretation happens in,
because in that domain a horizontally layered ground gives straight branches
whose slopes are the layer velocities, and every shot should trace out the
*same* branches regardless of where it was fired.

Read it for three things.

1. **Do the shots overlie one another?** They should. Shots that sit
   systematically above or below their neighbours mean a timing problem, a
   geometry problem, or ground that is not horizontally layered.
2. **Where does the curve bend?** That is the crossover, where a wave
   refracted along a deeper, faster layer overtakes the direct wave. One bend
   means two layers; two bends mean three. If you cannot see a bend, the data
   support one layer and no amount of fitting will change that.
3. **Do the forward-pointing and backward-pointing markers separate?** Colour
   runs light to dark with the source position along the line, and the marker
   points the way the wave travelled. A systematic split between the two at
   the same offset is a dipping interface; section 8 comes back to this.

In [ ]:
table = session.table()
print(f"{len(table)} picks from {table['shot'].nunique()} shots, "
      f"offsets {table['offset'].min():.1f} to {table['offset'].max():.1f} m")

fig, ax = plt.subplots(figsize=(10, 6.5))
plot_traveltimes(table, ax=ax)
ax.set_title("First breaks, all shots")
plt.show()

## 6. Slopes and intercepts to velocities and depths

For horizontal layers with velocities $v_0 < v_1 < \dots$ and thicknesses
$h_0, h_1, \dots$, the wave refracted along the top of layer $k$ arrives at

$$t_k(x) \;=\; \frac{x}{v_k} \;+\; \underbrace{\sum_{j<k}
   \frac{2 h_j \sqrt{v_k^2 - v_j^2}}{v_j v_k}}_{\text{intercept } t_{i,k}}$$

which is a **straight line** in $x$: slope $1/v_k$, intercept $t_{i,k}$. The
direct wave is the case $k = 0$, a line through the origin with slope $1/v_0$.
The first arrival at any offset is whichever of these lines is lowest there,
so the observed curve is a sequence of straight segments that get flatter
with distance.

So the method is: split the picks at the bends, fit a line to each segment,
and read off

$$v_k = \frac{1}{\text{slope}_k}, \qquad
  h_0 = \frac{t_{i,1}\, v_0 v_1}{2\sqrt{v_1^2 - v_0^2}},$$

with each deeper thickness following from its own intercept once the ones
above it are known. Two layers is one bend; three layers is two.

`fit_layers` does exactly this. Left to itself it tries every way of
splitting the picks into the requested number of segments and keeps the one
with the smallest misfit whose velocities increase with depth — which
automates the one step normally done by eye. Section 6b lets you overrule it.

**Assumptions you are signing up to:** planar horizontal interfaces,
homogeneous layers, velocity increasing with depth, everything on one
straight line at the surface. A layer that is slower than the one above it
produces no head wave and is invisible to this method — not fitted badly,
*invisible* — and its thickness is silently absorbed into the layers around
it.

In [ ]:
N_LAYERS = 2   # 2 or 3. Section 7 will tell you whether 3 is supported.

model = fit_layers(table["offset"], table["time"], n_layers=N_LAYERS)
print(model)
display(model.summary().round(2))

print(f"\ncrossover distance(s): "
      f"{', '.join(f'{x:.1f} m' for x in np.atleast_1d(model.crossovers))}")
print(f"direct-wave intercept if left unconstrained: "
      f"{1e3 * model.metadata['direct_intercept_s']:.2f} ms "
      f"(should be near zero -- a large value means a trigger-time error)")

### 6b. Put the bend where *you* think it is

The automatic split minimises misfit, which is not the same as being right:
scattered picks near the crossover let it place the bend a few metres either
way at almost no cost in misfit, and on a noisy far branch it can latch onto
the wrong bend entirely.

Drag the crossover to where you read it off the graph in section 5 and watch
what the velocities and the depth do. **How far you can move it before the
fit visibly worsens is your uncertainty on the depth** — a more honest error
bar than anything a single fit reports. Set `USE_MY_CROSSOVERS = True` to
carry your choice into the rest of the notebook.

In [ ]:
import ipywidgets as widgets

USE_MY_CROSSOVERS = False   # True keeps the slider values for sections 7-9

auto = np.atleast_1d(model.crossovers).astype(float)
x_lo, x_hi = float(table["offset"].min()), float(table["offset"].max())
manual = {}

def _try_crossovers(*cuts):
    fig, ax = plt.subplots(figsize=(9.5, 5.5))
    try:
        trial = fit_layers(table["offset"], table["time"],
                           n_layers=N_LAYERS, crossovers=list(cuts))
    except ValueError as exc:
        ax.text(0.5, 0.5, f"no fit here:\n{exc}", ha="center", va="center",
                wrap=True, fontsize=9, transform=ax.transAxes)
        ax.set_axis_off(); plt.show()
        manual.clear()
        return
    manual["model"] = trial
    plot_traveltimes(table, ax=ax, model=trial, legend=False)
    ax.set_title(
        "  ".join(f"V{i + 1} = {v:.0f} m/s" for i, v in enumerate(trial.velocities))
        + "   |   " + "  ".join(f"depth {d:.1f} m" for d in trial.depths)
        + f"   |   RMS {1e3 * trial.rms:.2f} ms", fontsize=10)
    plt.show()

sliders = [widgets.FloatSlider(value=float(auto[k]), min=x_lo, max=x_hi, step=0.25,
                               description=f"crossover {k + 1} (m)",
                               continuous_update=False,
                               style={"description_width": "130px"},
                               layout=widgets.Layout(width="520px"))
           for k in range(N_LAYERS - 1)]
display(widgets.interactive(_try_crossovers, **{f"c{k}": s
                                                for k, s in enumerate(sliders)}))

In [ ]:
if USE_MY_CROSSOVERS and "model" in manual:
    model = manual["model"]
    print("using your crossovers:", np.round(np.atleast_1d(model.crossovers), 2))
else:
    print("using the automatic split")
print(model)

## 7. Model against data

The fit is a claim: *this* stack of layers would have produced *these* travel
times. The plot below tests the claim. The top panel puts the modelled first
arrival over the picks; the bottom shows what is left over, observed minus
modelled, in milliseconds.

**Residuals are where the geology and the mistakes both live.** What to look
for, in order:

- **A few points far from zero, one shot, adjacent channels.** Mispicks. Go
  back to section 4, find them, fix them. Re-run from section 5.
- **A trend with offset** — residuals sloping across the whole graph. The
  velocities are wrong, usually because a branch boundary is in the wrong
  place. Try section 6b.
- **A shot sitting wholly above or below zero** while the others straddle it.
  That shot has a timing problem, or the ground under it is not what the
  single horizontal model says.
- **A smooth curve rather than scatter.** Real velocity gradients. The ground
  is not layers, it is a gradient, and no number of straight branches will
  fit it.

Scatter that looks like noise, of about the size of your picking precision
(a millisecond or two on hammer data), is the good outcome. It means the
model has used everything the data contain.

In [ ]:
fig, (ax_t, ax_r) = plt.subplots(2, 1, figsize=(10, 9.5), sharex=True,
                                 gridspec_kw={"height_ratios": [3, 1]})
plot_traveltimes(table, ax=ax_t, model=model, residual_ax=ax_r)
ax_t.set_title(f"{model.n_layers}-layer model against {len(table)} picks")
fig.tight_layout()
plt.show()

In [ ]:
# Residuals shot by shot: the fastest way to find the one record that is
# dragging the fit. Each panel is one shot; the band is +/- 2 ms, about the
# precision of a careful hand pick on hammer data.
resid = table.assign(resid_ms=1e3 * (table["time"] - model.predict(table["offset"])))
order = resid.groupby("shot")["source_x"].first().sort_values().index
ncols = 4
nrows = -(-len(order) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols, 2.4 * nrows),
                         sharex=True, sharey=True, squeeze=False)
for ax, shot in zip(axes.ravel(), order):
    part = resid[resid["shot"] == shot]
    ax.axhspan(-2, 2, color="#cde2fb", alpha=0.6, lw=0)
    ax.axhline(0, color="#0b0b0b", lw=0.8)
    ax.plot(part["offset"], part["resid_ms"], "o", ms=4, color="#256abf",
            mec="w", mew=0.5)
    ax.set_title(f"{shot}  ({part['source_x'].iloc[0]:+.1f} m)   "
                 f"mean {part['resid_ms'].mean():+.1f} ms", fontsize=8)
for ax in axes.ravel()[len(order):]:
    ax.axis("off")
fig.supxlabel("offset (m)"); fig.supylabel("obs - model (ms)")
fig.tight_layout()
plt.show()

worst = resid.reindex(resid["resid_ms"].abs().sort_values(ascending=False).index)
print("The ten picks furthest from the model -- start here when re-picking:")
print(worst.head(10)[["shot", "channel", "offset", "time", "quality",
                      "resid_ms"]].round(3).to_string(index=False))

## 8. Two layers or three?

Adding a layer always lowers the misfit — a three-segment line fits anything
a two-segment line fits, and more. So a smaller RMS is **not** evidence for
the third layer. The questions that are evidence:

1. **Does the deepest branch have enough picks over enough offset to define a
   slope?** A branch spanning the last few metres of the spread has a slope
   determined by scatter.
2. **Is the deepest velocity physically possible?** Above about 6000 m/s you
   are past nearly every rock that occurs near the surface. A fit returning
   tens of thousands of m/s is reporting a branch that is flat because it has
   no information, not a very fast layer.
3. **Does the extra interface fall inside the depth this spread can see?**
   Roughly a quarter to a third of the spread length, from section 2.

The cell below fits both and applies those checks. On a short spread the
usual and correct answer is two layers.

In [ ]:
MAX_PLAUSIBLE_V = 6000.0   # m/s -- faster than nearly any near-surface rock

comparison = []
fits = {}
for n in (2, 3):
    try:
        trial = fit_layers(table["offset"], table["time"], n_layers=n)
    except ValueError as exc:
        comparison.append({"layers": n, "verdict": f"no valid fit: {exc}"})
        continue
    fits[n] = trial
    branch = trial.branch_of()
    deepest = trial.offsets[branch == n - 1]
    span = float(deepest.max() - deepest.min()) if deepest.size else 0.0
    problems = []
    if deepest.size < 5:
        problems.append(f"deepest branch has only {deepest.size} picks")
    if span < 0.2 * float(table['offset'].max()):
        problems.append(f"deepest branch spans only {span:.1f} m")
    if trial.velocities[-1] > MAX_PLAUSIBLE_V:
        problems.append(f"V{n} = {trial.velocities[-1]:.0f} m/s is not a rock")
    if trial.depths[-1] > spread / 3:
        problems.append(f"deepest interface at {trial.depths[-1]:.1f} m is below "
                        f"what a {spread:.0f} m spread resolves")
    comparison.append({
        "layers": n,
        "velocities_m_s": ", ".join(f"{v:.0f}" for v in trial.velocities),
        "depths_m": ", ".join(f"{d:.1f}" for d in trial.depths),
        "rms_ms": round(1e3 * trial.rms, 2),
        "deepest_branch_picks": int(deepest.size),
        "verdict": "supported" if not problems else "; ".join(problems),
    })
display(pd.DataFrame(comparison))
print("A lower RMS for three layers is expected and proves nothing. Read the "
      "verdict column instead.")

In [ ]:
# The two fits drawn over the same picks. If the third layer is real, it
# follows a visible bend; if it is not, it is a line fitted to scatter.
assert fits, "neither a two- nor a three-layer fit succeeded; see section 7"
fig, axes = plt.subplots(1, len(fits), figsize=(6.5 * len(fits), 5.5),
                         sharey=True, squeeze=False)
for ax, (n, trial) in zip(axes[0], sorted(fits.items())):
    plot_traveltimes(table, ax=ax, model=trial, legend=False)
    ax.set_title(f"{n} layers - RMS {1e3 * trial.rms:.2f} ms\n"
                 + " / ".join(f"{v:.0f}" for v in trial.velocities) + " m/s",
                 fontsize=10)
fig.tight_layout()
plt.show()

## 9. Is the ground really horizontally layered?

Everything above assumed flat interfaces. The test is in the data you already
have, and it is the reason a line is shot from **both** ends.

Over a dipping interface, a shot fired downdip measures an apparent refractor
velocity that is too slow, and one fired updip measures one that is too fast.
Only the **off-end** shots — those fired from outside the spread — can be
used: a shot in the middle of the line sends waves both ways at once, so its
offset axis folds the two directions on top of each other and its apparent
velocity is a mixture of both.

If the two apparent velocities agree within their uncertainty, horizontal
layers are a fair assumption and the model from section 6 stands. If they
differ, the true velocity is close to their harmonic mean

$$v_2 \approx \frac{2 v_d v_u}{v_d + v_u}, \qquad
  \delta \approx \tfrac{1}{2}\left[\arcsin\frac{v_0}{v_d}
                                 - \arcsin\frac{v_0}{v_u}\right]$$

with $v_d$ downdip (slower) and $v_u$ updip (faster), and $\delta$ the dip.
Treat that as a **diagnostic, not a model** — a proper dipping-interface
inversion is not implemented in `shallowgeo` yet, and these formulas assume a
single plane interface and one shot from each end.

In [ ]:
x0, x1 = table["receiver_x"].min(), table["receiver_x"].max()
off_end = table[(table["source_x"] < x0 - 0.5) | (table["source_x"] > x1 + 0.5)]

if off_end.empty or off_end["source_x"].nunique() < 2:
    print("No pair of off-end shots, so the dip cannot be checked from these\n"
          "data. Next time, fire one shot beyond each end of the spread --\n"
          "it costs two hammer blows and it is the only way to find out.")
    apparent = {}
else:
    end = np.where(off_end["source_x"] < x0, "near end", "far end")
    apparent = {}
    for name, part in off_end.groupby(end):
        fit = fit_layers(part["offset"], part["time"], n_layers=2)
        apparent[name] = fit
        print(f"{name:9s} shots {sorted(part['shot'].unique())}: "
              f"V1 = {fit.velocities[0]:6.0f} m/s   "
              f"V2 apparent = {fit.velocities[1]:7.0f} m/s   "
              f"depth {fit.depths[0]:5.1f} m   RMS {1e3 * fit.rms:.2f} ms")

if len(apparent) == 2:
    v_pair = sorted(f.velocities[1] for f in apparent.values())
    v_d, v_u = v_pair
    v0 = np.mean([f.velocities[0] for f in apparent.values()])
    v2 = 2 * v_d * v_u / (v_d + v_u)
    if v0 < v_d:
        dip = 0.5 * (np.arcsin(v0 / v_d) - np.arcsin(v0 / v_u))
        print(f"\ndowndip {v_d:.0f} m/s, updip {v_u:.0f} m/s "
              f"-> true V2 about {v2:.0f} m/s, dip about "
              f"{np.degrees(dip):.1f} degrees")
    print(f"the two apparent velocities differ by a factor of {v_u / v_d:.2f}")
    print("\nBefore reading dip into that: a refractor branch confined to the far\n"
          "end of a short spread has a badly determined slope anyway. Quote V2 to\n"
          "one significant figure, and believe a dip only if it is larger than the\n"
          "spread between repeat shots fired from the same position.")

## 10. Write it up

Fill this in from the numbers above. A result without its limits is not a
result: the depth from a refraction line is usually good to a metre or so,
the shallow velocity to a few percent, and the refractor velocity to one
significant figure at best on a short spread.

In [ ]:
lines = [
    f"LINE            {len(rec_x)} geophones, {spread:.1f} m spread, "
    f"{np.diff(np.sort(rec_x)).min():.2f} m spacing",
    f"SHOTS           {len(shots)} gathers from {len(files_found)} files"
    + (f", excluded {EXCLUDE}" if EXCLUDE else ""),
    f"PICKS           {len(table)} used of {len(session.table(used_only=False))}, "
    f"{sum(int(p['edited'].sum()) for p in session.picks.values())} hand-edited, "
    f"picker '{session.settings['method']}' bounded at "
    f"{1e3 * session.settings['max_time']:.0f} ms",
    f"MODEL           {model.n_layers} layers, "
    + ", ".join(f"V{i + 1} = {v:.0f} m/s" for i, v in enumerate(model.velocities)),
    "INTERFACES      " + ", ".join(f"{d:.1f} m" for d in model.depths),
    f"MISFIT          RMS {1e3 * model.rms:.2f} ms",
    f"RESOLVED TO     about {spread / 3:.0f} m depth "
    f"({spread:.0f} m spread); anything below that is extrapolation",
]
if len(apparent) == 2:
    lines.append("DIP CHECK       apparent V2 "
                 + " vs ".join(f"{f.velocities[1]:.0f}" for f in apparent.values())
                 + " m/s from the two ends")
print("\n".join(lines))
print("\nINTERPRETATION  <- what are these layers? soil over weathered rock? "
      "water table?\nCONFIDENCE      <- how far can the crossover move before "
      "the fit worsens (section 6b)?")

## What to do differently next time

Most of what limits the answer above was decided in the field, not here.

**Shoot a longer spread than the depth you care about.** The refractor branch
has to be long enough for its slope to mean something. A spread three to four
times your target depth is the usual rule; on a spread barely longer than the
depth, the deep velocity is unconstrained no matter how carefully you pick.

**Shoot off both ends, well outside the spread.** Without a forward and
reverse pair there is no check on the horizontal-layer assumption at all, and
section 9 has nothing to work with. Two extra hammer blows.

**Repeat one shot.** Fire the same position twice, an hour apart. The spread
between the two answers is the only honest error bar you will get, and it
costs nothing.

**Stack more blows at long offset.** The far traces are where the picks are
worst and where the refractor velocity comes from. Five or ten blows stacked
at the far offsets buy more than anything you can do in processing.

**Write the geometry down on paper.** Source positions, geophone spacing,
which file is which shot, and the units the instrument was set to. Headers
are often wrong or empty, and a field note is the only way to catch it.